# MS-ILA G1 / E1 — Colab Pro end to end

Chọn **Runtime → Change runtime type → GPU** (L4 bạn đang dùng), rồi chạy lần lượt các cell.
Notebook này đã nhúng snapshot code G1 của repo: chỉ cần mở file `.ipynb`, không cần chờ GitHub được cập nhật.

Luồng chạy: **mount Drive → copy `.tar.gz` về `/content` → giải nén local → copy pretrained weights local → kiểm tra dữ liệu và GPU → smoke → Overfit-16 → train → synthetic DEV/native inference → lưu output về Drive**.

**G1 E1 chỉ dùng local 512×512, overlap 128, frozen DINOv3 patch16 và decoder.** Không có context/fusion/Adapter. `patch16` là patch của backbone; `TILE_SIZE` là crop ảnh đầu vào. Nếu bạn tăng `TILE_SIZE=768`, DINO nhận trực tiếp 768×768 và decoder xuất 768×768; không có bước 768→512. Size bin của synthetic vẫn tính bằng pixel native.

Trong pipeline multi-scale cũ, context 768→512 có thể làm suy yếu hoặc mất tiny defects do downsample. Notebook G1 này không thực hiện bước đó và không dùng thống kê TEST của `data_msila.ipynb` để hiệu chỉnh generator.

**Sửa đường dẫn dữ liệu/weights ở cell 2; đường dẫn output có cell 3 riêng.** Mặc định đường dẫn Drive lấy từ `notebooks/data_msila.ipynb`. Nếu archive chung không có tên category, đổi `ARCHIVE_PATTERNS` cho phù hợp.

Nguồn: [Colab — copy archive rồi giải nén local](https://research.google.com/colaboratory/faq.html#drive-quota), [official DINOv3](https://github.com/facebookresearch/dinov3), [official pretrained weights](https://ai.meta.com/resources/models-and-libraries/dinov3-downloads/).


In [ ]:
# @title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
from pathlib import Path
import os, sys, json, shutil, subprocess, time, hashlib
from IPython.display import display, Markdown, Image as DisplayImage


In [ ]:
# @title 2. INPUT + cấu hình GPU/training — sửa tại đây
# ĐƯỜNG DẪN INPUT — lấy theo data_msila.ipynb của bạn.
PROJECT_DRIVE = Path('/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9')
ARCHIVE_DIR = PROJECT_DRIVE / 'data'
WEIGHTS_DIR = PROJECT_DRIVE / 'weights'
CATEGORY = 'rice'
BACKBONE_NAME = 'dinov3_vits16'

# Một hoặc nhiều archive. Ví dụ ['*.tar.gz'] nếu archive chứa nhiều category.
ARCHIVE_PATTERNS = [f'*{CATEGORY}*.tar.gz']
# None: tự tìm đúng tên backbone trong weights/. Đặt Path(...) nếu muốn chỉ rõ file.
CHECKPOINT_ON_DRIVE = None

LOCAL_WORK = Path('/content/msila_g1')
LOCAL_ARCHIVES = LOCAL_WORK / 'archives'
LOCAL_DATA = LOCAL_WORK / 'dataset'
PROJECT_DIR = LOCAL_WORK / 'msila'
LOCAL_WEIGHTS = LOCAL_WORK / 'weights'
DINO_DIR = LOCAL_WORK / 'dinov3'
DINO_URL = 'https://github.com/facebookresearch/dinov3.git'
DINO_COMMIT = '6876159a11b4df116f30f667f8c9888617df0751'
VENV_DIR = LOCAL_WORK / 'venv'

# G1 chuẩn: giữ local 512 / overlap 128; không có context 768.
TILE_SIZE = 512
OVERLAP = 128
BATCH_SIZE = 4
EVAL_TILE_BATCH_SIZE = 4
NUM_WORKERS = 2
EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 2026
DEV_SEED = 17017
MAX_TRAIN_SOURCES = None  # None: toàn bộ TRAIN/good của category.
MAX_DEV_SOURCES = 4
TRAIN_VARIANTS_PER_IMAGE = 2
DEV_VARIANTS_PER_IMAGE = 6  # 4 sources × 6 = 24 tổ hợp type/placement/bin.
MAX_TRAIN_STEPS = None
MAX_TRAIN_MINUTES = 180  # kiểm tra giữa các step; evaluation/final export vẫn chạy.
OVERFIT_EPOCHS = 40
OVERFIT_MAX_STEPS = None
OVERFIT_MAX_MINUTES = 30
EXAMPLE_LIMIT = 8

RUN_SMOKE = True
RUN_OVERFIT16 = True
RUN_TRAIN = True
RESUME_SMOKE = False  # True nếu stage smoke bị ngắt sau khi đã có last.pt.
RESUME_OVERFIT16 = False  # True nếu muốn tiếp tục Overfit-16 từ last.pt.
RESUME_TRAIN = False  # True khi tiếp tục last.pt đã backup về Drive.
REQUIRE_OVERFIT_DECREASE = True
RUN_UNIT_TESTS = True

assert CATEGORY in ('can', 'fabric', 'fruit_jelly', 'rice', 'sheet_metal', 'vial', 'wallplugs', 'walnuts')
assert BACKBONE_NAME in ('dinov3_vits16', 'dinov3_vits16plus', 'dinov3_vitb16', 'dinov3_vith16plus')
assert TILE_SIZE > 0 and TILE_SIZE % 16 == 0 and 0 <= OVERLAP < TILE_SIZE
assert SEED != DEV_SEED and BATCH_SIZE >= 1 and EVAL_TILE_BATCH_SIZE >= 1
for folder in (LOCAL_WORK, LOCAL_ARCHIVES, LOCAL_DATA, LOCAL_WEIGHTS):
    folder.mkdir(parents=True, exist_ok=True)
print('Archive input:', ARCHIVE_DIR)
print('Local native data:', LOCAL_DATA)
print(f'Local tile={TILE_SIZE}, overlap={OVERLAP}; DINO patch=16; context branch=OFF')


In [ ]:
# @title 3. OUTPUT trên Drive — cell riêng để dễ thay đổi
# CHỈ CẤU HÌNH OUTPUT Ở CELL NÀY — bạn có thể thay từng folder theo Drive của mình.
# Giữ RUN_ID không đổi khi resume. Đổi RUN_ID cho thí nghiệm khác backbone/tile/seed.
RUN_ID = f'{BACKBONE_NAME}_local{TILE_SIZE}_seed{SEED}'
DRIVE_RUN_ROOT = PROJECT_DRIVE / 'outputs' / 'G1' / 'E1' / CATEGORY / RUN_ID
DRIVE_STAGE_DIRS = {
    'smoke': DRIVE_RUN_ROOT / 'smoke',
    'overfit16': DRIVE_RUN_ROOT / 'overfit16',
    'train': DRIVE_RUN_ROOT / 'train',
}
DRIVE_QA_DIR = DRIVE_RUN_ROOT / 'qa'
LOCAL_OUTPUT_BASE = LOCAL_WORK / 'outputs' / RUN_ID
LOCAL_OUTPUT_ROOTS = {name: LOCAL_OUTPUT_BASE / name for name in DRIVE_STAGE_DIRS}
# CLI tự thêm <category> vào output-root; Drive nhận nội dung folder category đó.
LOCAL_STAGE_DIRS = {name: root / CATEGORY for name, root in LOCAL_OUTPUT_ROOTS.items()}
LOCAL_QA_DIR = LOCAL_OUTPUT_BASE / 'qa'
# Chu kỳ backup checkpoint đã có + loss logs từ local lên Drive, tính bằng optimizer STEPS.
# Đổi thành 10 hoặc 20 tùy ý. Cuối stage / khi stop vẫn sync toàn bộ ngay.
DRIVE_SYNC_EVERY_STEPS = 20
assert type(DRIVE_SYNC_EVERY_STEPS) is int and DRIVE_SYNC_EVERY_STEPS > 0

for folder in (*DRIVE_STAGE_DIRS.values(), DRIVE_QA_DIR, LOCAL_QA_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print('Drive run:', DRIVE_RUN_ROOT)
for stage in DRIVE_STAGE_DIRS:
    print(f'{stage}: {LOCAL_STAGE_DIRS[stage]} -> {DRIVE_STAGE_DIRS[stage]}')
print('Train best.pt:', DRIVE_STAGE_DIRS['train'] / 'best.pt')
print('Synthetic DEV maps:', DRIVE_STAGE_DIRS['train'] / 'examples')
print(f'Drive sync mỗi {DRIVE_SYNC_EVERY_STEPS} optimizer steps (batch train); cuối stage sync toàn bộ.')


In [ ]:
# @title 4. Giải nén snapshot code G1 đã nhúng trong notebook
import base64, io, tarfile
CODE_BUNDLE_SHA256 = 'edb76aaad614f93cc3e969a69b6aae1d945fa9a5c208db694b7016ceaf12cd20'
CODE_BUNDLE_B64 = """
...
YA4A
"""
bundle = base64.b64decode(''.join(CODE_BUNDLE_B64.split()))
if hashlib.sha256(bundle).hexdigest() != CODE_BUNDLE_SHA256:
    raise RuntimeError('Snapshot code G1 bị thay đổi; dùng notebook nguyên bản.')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(bundle), mode='r:gz') as archive:
    for member in archive:
        relative = Path(member.name)
        if not member.isfile() or relative.is_absolute() or '..' in relative.parts:
            raise ValueError(f'Code archive member không hợp lệ: {member.name}')
        target = PROJECT_DIR / relative
        if not target.resolve().is_relative_to(PROJECT_DIR.resolve()):
            raise ValueError(f'Code path nằm ngoài PROJECT_DIR: {target}')
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.extractfile(member) as source, target.open('wb') as dest:
            shutil.copyfileobj(source, dest)
code_manifest = json.loads((PROJECT_DIR / 'colab_code_manifest.json').read_text())
for relative, expected in code_manifest['files'].items():
    actual = hashlib.sha256((PROJECT_DIR / relative).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Code checksum mismatch: {relative}')
(LOCAL_QA_DIR / 'code_snapshot.tar.gz').write_bytes(bundle)
(LOCAL_QA_DIR / 'code_manifest.json').write_text(json.dumps(code_manifest, indent=2))
print('Code G1 ready:', PROJECT_DIR, 'SHA256:', CODE_BUNDLE_SHA256)


In [ ]:
# @title 5. Môi trường CUDA + dependencies
# Venv riêng kế thừa CUDA PyTorch của Colab; không cài đè torch/torchvision.
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Chọn GPU ở Runtime > Change runtime type, rồi chạy lại notebook.')
if tuple(int(v) for v in torch.__version__.split('+')[0].split('.')[:2]) < (2, 6):
    raise RuntimeError(f'G1 cần PyTorch >=2.6; runtime hiện tại: {torch.__version__}')
gpu = torch.cuda.get_device_properties(0)
print(f'GPU: {gpu.name}, VRAM={gpu.total_memory / 2**30:.1f} GiB; torch={torch.__version__}')
if not (VENV_DIR / 'bin/python').is_file():
    # Kế thừa pip đã có trong Colab; không phụ thuộc ensurepip của runtime.
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', str(VENV_DIR)], check=True)
PYTHON = str(VENV_DIR / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', '--version'], check=True)
dependencies = [line for line in (PROJECT_DIR / 'requirements.txt').read_text().splitlines()
                if line.strip() and not line.lstrip().startswith(('#', 'torch'))]
requirements_path = LOCAL_WORK / 'requirements_colab.txt'
requirements_path.write_text('\n'.join(dependencies) + '\n')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)], check=True)
TRAIN_ENV = dict(os.environ, PYTHONUNBUFFERED='1', PYTEST_DISABLE_PLUGIN_AUTOLOAD='1',
                 MPLCONFIGDIR=str(LOCAL_WORK / 'matplotlib_cache'), TMPDIR=str(LOCAL_WORK / 'tmp'))
Path(TRAIN_ENV['TMPDIR']).mkdir(parents=True, exist_ok=True)
subprocess.run([PYTHON, '-c', 'import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))'],
               env=TRAIN_ENV, check=True)
print('Python training:', PYTHON)


In [ ]:
# @title 6. Helper copy và giải nén native good-only
# Helper I/O: copy atomic, SHA256, chỉ lấy TRAIN/good + VALIDATION/good.
import re
from pathlib import PurePosixPath

def sha256_file(path):
    with Path(path).open('rb') as source:
        return hashlib.file_digest(source, 'sha256').hexdigest()

def copy_atomic(source, dest):
    source, dest = Path(source), Path(dest)
    if not source.is_file():
        raise FileNotFoundError(source)
    if source.resolve() == dest.resolve():
        raise ValueError('Source và destination phải khác nhau')
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_file() and (dest.stat().st_size, dest.stat().st_mtime_ns) == (source.stat().st_size, source.stat().st_mtime_ns):
        return dest
    temporary = dest.with_name(dest.name + '.copying')
    shutil.copy2(source, temporary)
    temporary.replace(dest)
    return dest

def target_good_path(member_name, category, archive_name):
    path = PurePosixPath(member_name)
    if path.is_absolute() or '..' in path.parts or '\\' in member_name:
        raise ValueError(f'Tar path không hợp lệ: {member_name}')
    parts = path.parts
    lowered = [p.casefold() for p in parts]
    known_categories = {'can', 'fabric', 'fruit_jelly', 'rice', 'sheet_metal', 'vial', 'wallplugs', 'walnuts'}
    named_categories = known_categories.intersection(lowered)
    if named_categories and category not in named_categories:
        return None
    categoryless = not named_categories
    if categoryless and not re.search(r'(?:^|[_\-.])' + re.escape(category) + r'(?:[_\-.]|$)', archive_name.casefold()):
        return None
    aliases = {'train': 'train', 'training': 'train', 'validation': 'validation', 'val': 'validation', 'valid': 'validation'}
    splits = [(i, aliases[p]) for i, p in enumerate(lowered) if p in aliases]
    if len(splits) != 1:
        return None
    i, split = splits[0]
    if i + 2 >= len(parts) or lowered[i+1] != 'good':
        return None
    if not categoryless and category not in lowered[:i]:
        return None
    tail = parts[i+2:]
    mask_dirs = {'ground_truth', 'groundtruth', 'gt', 'mask', 'masks', 'label', 'labels', 'segmentation', 'segmentations'}
    if mask_dirs.intersection(p.casefold() for p in tail[:-1]):
        return None
    suffixes = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}
    if path.suffix.casefold() not in suffixes or path.stem.casefold().endswith(('_mask', '_gt', '_label')):
        return None
    return Path(category, split, 'good', *tail)

def extract_good_archive(local_archive, dataset_root, category):
    # Đọc archive LOCAL; không giải nén trên mounted Drive.
    dataset_root = Path(dataset_root)
    if dataset_root.resolve().is_relative_to(Path('/content/drive')):
        raise ValueError('Dataset phải được giải nén trên disk local Colab')
    count, expanded = 0, 0
    with tarfile.open(local_archive, 'r:gz') as archive:
        for member in archive:
            relative = target_good_path(member.name, category, Path(local_archive).name)
            if relative is None or member.isdir():
                continue
            if not member.isfile():
                raise ValueError(f'Ảnh trong archive phải là regular file: {member.name}')
            target = dataset_root / relative
            if not target.resolve().is_relative_to(dataset_root.resolve()):
                raise ValueError(f'Tar output ngoài dataset root: {target}')
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_name(target.name + '.extracting')
            with archive.extractfile(member) as source, temporary.open('wb') as dest:
                shutil.copyfileobj(source, dest, length=8*1024*1024)
            if temporary.stat().st_size != member.size:
                raise RuntimeError(f'Extract thiếu dữ liệu: {member.name}')
            if target.exists() and sha256_file(target) != sha256_file(temporary):
                temporary.unlink()
                raise ValueError(f'Hai archive chứa ảnh khác nhau tại cùng path: {relative}')
            temporary.replace(target)
            count += 1
            expanded += member.size
    return dict(images=count, extracted_bytes=expanded)


In [ ]:
# @title 7. Copy toàn bộ archive đã chọn Drive → Colab, rồi giải nén LOCAL
if not ARCHIVE_DIR.is_dir():
    raise FileNotFoundError(f'Sửa ARCHIVE_DIR ở cell 2: {ARCHIVE_DIR}')
archives = sorted({p for pattern in ARCHIVE_PATTERNS for p in ARCHIVE_DIR.glob(pattern) if p.is_file()})
if not archives:
    print('Archives đang có:', [p.name for p in ARCHIVE_DIR.glob('*.tar.gz')])
    raise FileNotFoundError('Không có archive khớp ARCHIVE_PATTERNS; sửa cell 2.')
required = sum(p.stat().st_size for p in archives)
free = shutil.disk_usage(LOCAL_WORK).free
print(f'Archives: {required/2**30:.2f} GiB; local free: {free/2**30:.2f} GiB')
if required > free:
    raise RuntimeError('Disk local không đủ cho archive; chọn archive của một category.')
local_archive_paths = []
for index, source in enumerate(archives, 1):
    started = time.monotonic()
    target = copy_atomic(source, LOCAL_ARCHIVES / source.name)
    local_archive_paths.append(target)
    print(f'[{index}/{len(archives)}] COPIED {source.name}: {time.monotonic()-started:.1f}s')

# Tính dung lượng GOOD cần giải nén trước khi ghi ảnh.
expanded_bytes = 0
for archive_path in local_archive_paths:
    with tarfile.open(archive_path, 'r:gz') as archive:
        for member in archive:
            if member.isfile() and target_good_path(member.name, CATEGORY, archive_path.name) is not None:
                expanded_bytes += member.size
if expanded_bytes > shutil.disk_usage(LOCAL_WORK).free:
    raise RuntimeError(f'Disk local không đủ để giải nén {expanded_bytes/2**30:.2f} GiB good images.')
archive_records = []
for index, archive_path in enumerate(local_archive_paths, 1):
    started = time.monotonic()
    record = dict(name=archive_path.name, sha256=sha256_file(archive_path),
                  **extract_good_archive(archive_path, LOCAL_DATA, CATEGORY))
    archive_records.append(record)
    print(f'[{index}/{len(local_archive_paths)}] EXTRACTED {record["images"]} good images: {time.monotonic()-started:.1f}s')
if not sum(r['images'] for r in archive_records):
    raise RuntimeError('Không tìm thấy category/train/good và category/validation/good trong archive đã chọn.')
(LOCAL_QA_DIR / 'archives_manifest.json').write_text(json.dumps(archive_records, indent=2))
print('Native data root:', LOCAL_DATA)
print('Chỉ TRAIN/good và VALIDATION/good đã được giải nén; TEST không tham gia G1.')


In [ ]:
# @title 8. Copy pretrained weights local + clone official DINOv3 đã pin commit
official_names = {
    'dinov3_vits16': 'dinov3_vits16_pretrain_lvd1689m-08c60483.pth',
    'dinov3_vits16plus': 'dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth',
    'dinov3_vitb16': 'dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth',
    'dinov3_vith16plus': 'dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth',
}
if CHECKPOINT_ON_DRIVE is not None:
    drive_checkpoint = Path(CHECKPOINT_ON_DRIVE)
else:
    exact = WEIGHTS_DIR / official_names[BACKBONE_NAME]
    candidates = sorted(WEIGHTS_DIR.glob(f'{BACKBONE_NAME}_pretrain_lvd1689m*.pth'))
    if exact.is_file():
        drive_checkpoint = exact
    elif len(candidates) == 1:
        drive_checkpoint = candidates[0]
    else:
        raise FileNotFoundError(f'Chỉ rõ CHECKPOINT_ON_DRIVE ở cell 2. Candidates: {candidates}')
LOCAL_CHECKPOINT = copy_atomic(drive_checkpoint, LOCAL_WEIGHTS / drive_checkpoint.name)
CHECKPOINT_SHA256 = sha256_file(LOCAL_CHECKPOINT)
if not (DINO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', DINO_URL, str(DINO_DIR)], check=True)
subprocess.run(['git', '-C', str(DINO_DIR), 'checkout', DINO_COMMIT], check=True)
actual_dino_commit = subprocess.check_output(['git', '-C', str(DINO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_dino_commit == DINO_COMMIT
subprocess.run([PYTHON, '-m', 'src.train.g1_e1', '--help'], cwd=PROJECT_DIR, env=TRAIN_ENV, check=True)
print('Local official checkpoint:', LOCAL_CHECKPOINT)
print('Checkpoint SHA256:', CHECKPOINT_SHA256)
print('Official DINO commit:', actual_dino_commit)


In [ ]:
# @title 9. Sinh YAML từ configs/g1_e1.yaml — cùng code path cho cả ba stage
import yaml, copy
base = yaml.safe_load((PROJECT_DIR / 'configs/g1_e1.yaml').read_text())
base['category'] = CATEGORY
base['backbone']['name'] = BACKBONE_NAME
base['backbone']['repo_dir'] = str(DINO_DIR)
base['backbone']['checkpoints'][BACKBONE_NAME] = str(LOCAL_CHECKPOINT)
base['synthetic_protocol'] = str(PROJECT_DIR / 'configs/full_scale_synthetic.yaml')
base['data'].update(root=str(LOCAL_DATA), tile_size=TILE_SIZE, overlap=OVERLAP, num_workers=NUM_WORKERS,
                    max_train_sources=MAX_TRAIN_SOURCES, max_dev_sources=MAX_DEV_SOURCES,
                    train_variants_per_image=TRAIN_VARIANTS_PER_IMAGE, dev_variants_per_image=DEV_VARIANTS_PER_IMAGE)
base['training'].update(seed=SEED, dev_seed=DEV_SEED, epochs=EPOCHS, batch_size=BATCH_SIZE,
                        learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
                        max_steps=MAX_TRAIN_STEPS, max_minutes=MAX_TRAIN_MINUTES)
base['evaluation'].update(tile_batch_size=EVAL_TILE_BATCH_SIZE, example_limit=EXAMPLE_LIMIT)
base['colab_provenance'] = dict(code_bundle_sha256=CODE_BUNDLE_SHA256, dino_commit=DINO_COMMIT,
                                checkpoint_sha256=CHECKPOINT_SHA256,
                                archive_sha256={r['name']: r['sha256'] for r in archive_records},
                                geometry='native_local_tiles_no_context_no_downsample', precision='fp32')
CONFIG_PATHS = {}
RAW_STAGE_CONFIGS = {}
config_dir = LOCAL_OUTPUT_BASE / 'configs'
config_dir.mkdir(parents=True, exist_ok=True)
for stage in DRIVE_STAGE_DIRS:
    config = copy.deepcopy(base)
    config['output_root'] = str(LOCAL_OUTPUT_ROOTS[stage])
    config['training']['mode'] = 'train' if stage == 'train' else stage
    if stage == 'smoke':
        config['training'].update(epochs=1, max_steps=2, max_minutes=10)
        config['data'].update(max_train_sources=2, max_dev_sources=1, dev_variants_per_image=3, num_workers=0)
        config['smoke'].update(epochs=1, max_steps=2, max_train_sources=2, max_dev_sources=1, dev_variants_per_image=3)
    elif stage == 'overfit16':
        config['training'].update(epochs=OVERFIT_EPOCHS, max_steps=OVERFIT_MAX_STEPS, max_minutes=OVERFIT_MAX_MINUTES)
        config['data'].update(max_train_sources=8, max_dev_sources=1, dev_variants_per_image=3, num_workers=0)
    path = config_dir / f'{stage}.yaml'
    path.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))
    CONFIG_PATHS[stage] = path
    RAW_STAGE_CONFIGS[stage] = config
print(yaml.safe_dump(RAW_STAGE_CONFIGS['train'], sort_keys=False, allow_unicode=True))


In [ ]:
# @title 10. Audit good-only, không leakage; visualize native + tiny synthetic
audit_script = r"""
from pathlib import Path
import json, os, platform, sys
import torch, yaml, numpy as np
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.train.g1_e1 import resolve_config, parse_args, check_assets, discover_sources
from src.data.loader import load_rgb_native
from src.data.synthetic_anomaly import NativeTinyDefectGenerator
config = resolve_config(parse_args(['--config', sys.argv[1]]))
out = Path(sys.argv[2]); out.mkdir(parents=True, exist_ok=True)
check_assets(config, 'cuda')
pools, manifest = discover_sources(config)
if len(pools['train']) < 8:
    raise ValueError('Overfit-16 cần ít nhất tám TRAIN/good sources khác nhau.')
report = dict(counts={role: len(rows) for role, rows in pools.items()}, source_manifest=manifest,
              selected_category=config['category'], dataset_root=config['data']['root'], test_used=False)
(out/'data_audit.json').write_text(json.dumps(report, indent=2))
environment = dict(python=sys.version, torch=torch.__version__, cuda=torch.version.cuda,
                   gpu=torch.cuda.get_device_name(0), vram_bytes=torch.cuda.get_device_properties(0).total_memory,
                   precision='fp32', platform=platform.platform())
(out/'environment.json').write_text(json.dumps(environment, indent=2))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, role in enumerate(('train', 'dev')):
    for col in range(4):
        record = pools[role][col % len(pools[role])]
        image = load_rgb_native(record.image_path)
        axes[row,col].imshow(image.permute(1,2,0).numpy())
        axes[row,col].set_title(f'{record.split}/good — {tuple(image.shape[-2:])}')
        axes[row,col].axis('off')
fig.tight_layout(); fig.savefig(out/'good_native_preview.png', dpi=130); plt.close(fig)
protocol = config['synthetic_protocol']; generator = NativeTinyDefectGenerator(protocol)
original = load_rgb_native(pools['train'][0].image_path)
cases = [('sub_patch','pinhole','sub_patch'),('small','texture','small'),('thin_scratch','thin_scratch','small'),('low_contrast','contamination','sub_patch')]
fig, axes = plt.subplots(4, 4, figsize=(12, 12)); records=[]
for row, (name, kind, size_bin) in enumerate(cases):
    if name == 'low_contrast':
        # Tìm seed có contrast nằm trong 1/8 thấp nhất của protocol, không sửa contrast_range.
        lo, hi = protocol['contrast_range']
        for seed in range(100, 1100):
            sample = generator(original, seed=seed, defect_type=kind, size_bin=size_bin, placement='interior')
            if sample.metadata['contrast'] <= lo + (hi-lo)/8:
                break
        else:
            raise RuntimeError('Không tìm được low-contrast seed trong protocol đã khóa.')
    else:
        sample = generator(original, seed=71+row, defect_type=kind, size_bin=size_bin, placement='interior')
    image = sample.image.permute(1,2,0).numpy(); mask=sample.mask[0].numpy()
    overlay=image.copy(); overlay[mask.astype(bool)] = .45*overlay[mask.astype(bool)] + .55*np.array([1.,0.,0.])
    x0,y0,x1,y1=sample.metadata['bbox_xyxy']; h,w=mask.shape
    span=max(96,x1-x0+20,y1-y0+20)
    left=max(0,min(w-span,(x0+x1-span)//2)); top=max(0,min(h-span,(y0+y1-span)//2))
    for ax, value, title in zip(axes[row], (original.permute(1,2,0).numpy(),image,mask,overlay), ('Original','Synthetic','Exact mask','GT overlay')):
        value=value[top:top+span,left:left+span]
        ax.imshow(value, **({'cmap':'gray','vmin':0,'vmax':1} if value.ndim==2 else {}))
        ax.set_title(f'{name}: {title}\narea={sample.metadata["area"]}px; contrast={sample.metadata["contrast"]:.3f}'); ax.axis('off')
    records.append(dict(case=name, source=pools['train'][0].image_path, **sample.metadata))
fig.tight_layout(); fig.savefig(out/'synthetic_native_gallery.png', dpi=140); plt.close(fig)
(out/'synthetic_preview_metadata.json').write_text(json.dumps(records, indent=2))
print(json.dumps(dict(counts=report['counts'], dataset_root=report['dataset_root'], test_used=False), indent=2))
"""
subprocess.run([PYTHON, '-c', audit_script, str(CONFIG_PATHS['train']), str(LOCAL_QA_DIR)],
               cwd=PROJECT_DIR, env=TRAIN_ENV, check=True)
display(DisplayImage(filename=str(LOCAL_QA_DIR / 'good_native_preview.png')))
display(DisplayImage(filename=str(LOCAL_QA_DIR / 'synthetic_native_gallery.png')))


L4: bắt đầu với **local 512, batch 4**. Cell tiếp theo đo một train step thật với pretrained weights, rồi giải phóng model của phép đo. Nếu còn nhiều VRAM, bạn có thể tăng `BATCH_SIZE` và chạy lại phép đo trước một **RUN_ID mới**. Tile local 768/1024 là lựa chọn geometry khác, không phải context 768 của multi-scale; giữ overlap tương ứng và không resize xuống 512. A100 hoặc GPU lớn hơn vẫn cần đo ở runtime thực tế.

Notebook giữ precision FP32 theo trainer G1 hiện tại, không bật AMP riêng để tạo một train path khác.


In [ ]:
# @title 11. Đo VRAM bằng train step thật + kiểm tra frozen/gradient
probe_script = r"""
from pathlib import Path
import json, sys
import torch
from src.train.g1_e1 import resolve_config, parse_args, discover_sources, module_sha256
from src.data.loader import G1NativeDataset, G1TileDataset, g1_tile_collate
from src.models.dinov3_extractor import DINOv3FeatureExtractor
from src.models.msila import E1
from src.train.optimizer import build_optimizer
from src.train.overfit16 import Overfit16Trainer, seed_everything
from src.losses.anomaly_loss import AnomalySegmentationLoss
cfg=resolve_config(parse_args(['--config',sys.argv[1]])); seed_everything(cfg['training']['seed'])
pools,_=discover_sources(cfg)
native=G1NativeDataset(pools['train'][:2],cfg['synthetic_protocol'],seed=cfg['training']['seed'],role='train',variants=1)
tiles=G1TileDataset(native,tile_size=cfg['data']['tile_size'],overlap=cfg['data']['overlap'])
batch=g1_tile_collate([tiles[i % len(tiles)] for i in range(cfg['training']['batch_size'])])
ext=DINOv3FeatureExtractor(cfg['backbone']['repo_dir'],cfg['backbone']['weights'],model_name=cfg['backbone']['name'],feature_mode='deepest',check_finite=True)
model=E1(ext,**cfg['decoder']).cuda()
optimizer,_=build_optimizer({'decoder':model.decoder},frozen_modules={'backbone':ext},learning_rate=cfg['training']['learning_rate'],weight_decay=cfg['training']['weight_decay'])
trainer=Overfit16Trainer(model=model,criterion=AnomalySegmentationLoss(**cfg['loss']),optimizer=optimizer,device='cuda',frozen_modules={'backbone':ext})
backbone_before=module_sha256(ext); decoder_before=module_sha256(model.decoder)
torch.cuda.reset_peak_memory_stats()
step=trainer.train_step(batch,step=1,epoch=0)
torch.cuda.synchronize()
assert ext.backbone_is_frozen() and not ext.backbone.training
assert all(p.grad is None for p in ext.parameters())
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.decoder.parameters())
assert module_sha256(ext)==backbone_before and module_sha256(model.decoder)!=decoder_before
result=dict(status='PASS',backbone=cfg['backbone']['name'],channels=ext.out_channels,depth=ext.depth,selected_blocks=list(ext.blocks),patch_size=ext.patch_size,
            tile_size=cfg['data']['tile_size'],dense_hw=[cfg['data']['tile_size']//ext.patch_size]*2,batch_size=cfg['training']['batch_size'],
            gpu=torch.cuda.get_device_name(0),peak_allocated_gib=torch.cuda.max_memory_allocated()/2**30,
            peak_reserved_gib=torch.cuda.max_memory_reserved()/2**30,total_vram_gib=torch.cuda.get_device_properties(0).total_memory/2**30,
            frozen_unchanged=True,decoder_updated=True,context_used=False,downsample_to_512=False)
Path(sys.argv[2]).write_text(json.dumps(result,indent=2)); print(json.dumps(result,indent=2))
"""
subprocess.run([PYTHON, '-c', probe_script, str(CONFIG_PATHS['train']), str(LOCAL_QA_DIR / 'gpu_probe.json')],
               cwd=PROJECT_DIR, env=TRAIN_ENV, check=True)
if RUN_UNIT_TESTS:
    test_command = [PYTHON, '-m', 'pytest', '-q', 'tests/test_g1_e1.py', 'tests/test_dinov3_extractor.py',
                    'tests/test_tiling.py', 'tests/test_anomaly_loss.py', 'tests/test_checkpoint.py', 'tests/test_resume.py',
                    '--junitxml=' + str(LOCAL_QA_DIR / 'g1_pytest.xml')]
    subprocess.run(test_command, cwd=PROJECT_DIR, env=TRAIN_ENV, check=True)


In [ ]:
# @title 12. Sync output về Drive + restore/resume helpers
# Checkpoint/log được ghi trên disk local; chỉ backup Drive theo chu kỳ ở cell OUTPUT.
def sync_checkpoint(source, destination):
    source, destination = Path(source), Path(destination)
    sidecar = source.with_suffix(source.suffix + '.sha256')
    if not source.is_file() or not sidecar.is_file():
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    target_sidecar = destination.with_suffix(destination.suffix + '.sha256')
    for attempt in range(3):
        expected = sidecar.read_text().split()[0]
        if destination.is_file() and target_sidecar.is_file() and target_sidecar.read_text().split()[0] == expected and sha256_file(destination) == expected:
            return
        temporary = destination.with_name(destination.name + '.copying')
        shutil.copyfile(source, temporary)
        if sha256_file(temporary) == expected:
            temporary.replace(destination)
            temp_sidecar = target_sidecar.with_name(target_sidecar.name + '.copying')
            temp_sidecar.write_text(f'{expected}  {destination.name}\n')
            temp_sidecar.replace(target_sidecar)
            return
    raise RuntimeError(f'Checkpoint thay đổi khi sync hoặc copy lỗi: {source}')

def sync_outputs(stage, full=False):
    source, destination = LOCAL_STAGE_DIRS[stage], DRIVE_STAGE_DIRS[stage]
    if not source.is_dir():
        return
    for name in ('last.pt', 'best.pt'):
        sync_checkpoint(source / name, destination / name)
    critical = ('resolved_config.yaml', 'launch_config.yaml', 'train.log', 'console.log', 'metrics.json',
                'evaluation_metrics.json', 'qa_report.json', 'failure.json')
    files = sorted(source.rglob('*')) if full else [source / name for name in critical]
    for path in files:
        if not path.is_file() or path.suffix == '.pt' or path.name.endswith(('.pt.sha256', '.copying')):
            continue
        copy_atomic(path, destination / path.relative_to(source))

def restore_outputs(stage):
    source, destination = DRIVE_STAGE_DIRS[stage], LOCAL_STAGE_DIRS[stage]
    destination.mkdir(parents=True, exist_ok=True)
    if not source.is_dir():
        return
    for name in ('last.pt', 'best.pt'):
        sync_checkpoint(source / name, destination / name)
    for path in sorted(source.rglob('*')):
        if path.is_file() and path.suffix != '.pt' and not path.name.endswith(('.pt.sha256','.copying')):
            copy_atomic(path, destination / path.relative_to(source))

def scientific_identity(config):
    config = copy.deepcopy(config)
    for key in ('epochs', 'max_steps', 'max_minutes'):
        config['training'].pop(key, None)
    return config

def launch_stage(stage, resume=False):
    restore_outputs(stage)
    directory = LOCAL_STAGE_DIRS[stage]
    launch_config = directory / 'launch_config.yaml'
    current = RAW_STAGE_CONFIGS[stage]
    if launch_config.is_file():
        previous = yaml.safe_load(launch_config.read_text())
        if scientific_identity(previous) != scientific_identity(current):
            raise RuntimeError(f'{stage}: cấu hình hoặc input đã đổi. Chọn RUN_ID mới ở cell OUTPUT.')
    metric_path = directory / 'metrics.json'
    if (directory / 'best.pt').is_file() and metric_path.is_file() and not resume:
        previous_result = json.loads(metric_path.read_text())
        if previous_result.get('status') == 'PASS' and launch_config.is_file():
            print(f'{stage}: dùng lại run đã hoàn tất trong đúng RUN_ID; muốn tiếp tục đặt RESUME_{stage.upper()}=True.')
            return previous_result
    if (directory / 'last.pt').is_file() and not resume:
        raise RuntimeError(f'{stage}: có checkpoint nhưng chưa hoàn tất. Đặt RESUME_{stage.upper()}=True ở cell 2 hoặc RUN_ID mới.')
    if resume and not (directory / 'last.pt').is_file():
        raise FileNotFoundError(f'Không có last.pt để resume: {DRIVE_STAGE_DIRS[stage]}')
    launch_config.write_text(yaml.safe_dump(current, sort_keys=False, allow_unicode=True))
    command = [PYTHON, '-m', 'src.train.g1_e1', '--config', str(CONFIG_PATHS[stage]), '--category', CATEGORY, '--device', 'cuda']
    if stage == 'smoke': command += ['--smoke']
    elif stage == 'overfit16': command += ['--overfit16']
    if resume: command += ['--resume', str(directory / 'last.pt')]
    print('Running:', subprocess.list2cmdline(command))
    process = subprocess.Popen(command, cwd=PROJECT_DIR, env=TRAIN_ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        with (directory / 'console.log').open('a') as console:
            for line in process.stdout:
                print(line, end=''); console.write(line); console.flush()
                # Chỉ log của optimizer step được đếm; log checkpoint/DEV không gây thêm Drive I/O.
                logged_step = re.search(r'\bepoch=\d+\s+step=(\d+)\s+loss=', line)
                if logged_step and int(logged_step.group(1)) % DRIVE_SYNC_EVERY_STEPS == 0:
                    sync_outputs(stage)
        returncode = process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill(); process.wait()
        raise
    finally:
        sync_outputs(stage, full=True)
    if returncode:
        raise RuntimeError(f'{stage} exit={returncode}; xem train.log/console.log ở {DRIVE_STAGE_DIRS[stage]}')
    return json.loads(metric_path.read_text())

for path in LOCAL_QA_DIR.iterdir():
    if path.is_file(): copy_atomic(path, DRIVE_QA_DIR / path.name)
print('QA/snapshot saved:', DRIVE_QA_DIR)


In [ ]:
# @title 13. Smoke — 2 optimizer steps, cùng CLI E1
if RUN_SMOKE:
    smoke_result = launch_stage('smoke', resume=RESUME_SMOKE)
    assert smoke_result['status'] == 'PASS'
    print('SMOKE:', smoke_result['status'])
else:
    print('Smoke được tắt ở cell 2.')


In [ ]:
# @title 14. Overfit-16 — tám normal + tám synthetic cố định, architecture QA
if RUN_OVERFIT16:
    overfit_result = launch_stage('overfit16', resume=RESUME_OVERFIT16)
    print('Initial loss:', overfit_result['initial_overfit_loss'])
    print('Final loss:', overfit_result['final_overfit_loss'])
    print('Loss decreased:', overfit_result['overfit_loss_decreased'])
    if REQUIRE_OVERFIT_DECREASE and not overfit_result['overfit_loss_decreased']:
        raise RuntimeError('Overfit-16 chưa giảm loss. Kiểm tra log/cấu hình; chưa đủ bằng chứng học trước full train.')
else:
    print('Overfit-16 được tắt; acceptance cuối sẽ ghi NOT RUN nếu chưa có evidence.')


In [ ]:
# @title 15. Train E1 trên TRAIN/good + synthetic DEV, backup Drive theo chu kỳ
if RUN_TRAIN:
    train_result = launch_stage('train', resume=RESUME_TRAIN)
    display(train_result)
else:
    print('Full training được tắt ở cell 2.')


Resume sau khi Colab bị ngắt: giữ nguyên input, tile/backbone/seed và **RUN_ID**; đặt `RESUME_TRAIN=True`, rồi chạy lại các cell chuẩn bị và cell train. Nếu smoke hoặc Overfit-16 bị ngắt sau checkpoint, dùng `RESUME_SMOKE` hoặc `RESUME_OVERFIT16` tương ứng. Checkpoint được restore từ Drive về đúng local path trước khi gọi CLI `--resume`. Có thể tăng `EPOCHS`, `MAX_TRAIN_STEPS` hoặc `MAX_TRAIN_MINUTES`; các tham số khoa học khác được khóa bởi runner. Các stage QA đã hoàn tất sẽ được dùng lại nếu cấu hình/input khớp.

Checkpoint được ghi trên disk local sau đánh giá DEV ở cuối epoch hoặc khi chạm budget. Loss/BCE/Dice/lr vẫn được log local mỗi step. Notebook backup checkpoint đã có và logs lên Drive mỗi `DRIVE_SYNC_EVERY_STEPS` optimizer steps ở cell 3 (mặc định 20 batch train), theo global step nên tiếp tục đúng nhịp khi resume, và sync toàn bộ ở cuối stage / khi dừng qua Python exception. Nếu chưa hết epoch, checkpoint được backup là bản local gần nhất; chu kỳ Drive không tạo thêm checkpoint local. `metrics.json` và loss curve được xuất khi stage hoàn tất. Một runtime bị thu hồi trước checkpoint đầu tiên chưa có trạng thái train để resume. `max_minutes` kiểm tra giữa các step; DEV/final export có thể kéo dài thêm.


In [ ]:
# @title 16. Native full-resolution inference, AU-PRO@0.05, P99 và visualize
restore_outputs('train')
train_dir = LOCAL_STAGE_DIRS['train']
if not (train_dir / 'best.pt').is_file():
    print('Không có best.pt; chưa chạy native DEV evaluation.')
else:
    command = [PYTHON, '-m', 'src.train.g1_e1', '--config', str(CONFIG_PATHS['train']),
               '--category', CATEGORY, '--device', 'cuda', '--evaluate', str(train_dir / 'best.pt')]
    try:
        subprocess.run(command, cwd=PROJECT_DIR, env=TRAIN_ENV, check=True)
    finally:
        sync_outputs('train', full=True)
    display(json.loads((train_dir / 'evaluation_metrics.json').read_text()))
    for path in [train_dir / 'loss_curve.png', *sorted((train_dir / 'evaluation/examples').glob('*/comparison.png'))[:4]]:
        if path.is_file(): display(DisplayImage(filename=str(path)))


In [ ]:
# @title 17. G1 acceptance và vị trí output trên Drive
# Chỉ báo PASS từ evidence của pretrained run thật trên Colab, không điền metric giả.
def read_json(path):
    return json.loads(Path(path).read_text()) if Path(path).is_file() else {}

probe = read_json(LOCAL_QA_DIR / 'gpu_probe.json')
overfit = read_json(DRIVE_STAGE_DIRS['overfit16'] / 'metrics.json')
training = read_json(DRIVE_STAGE_DIRS['train'] / 'metrics.json')
evaluation = read_json(DRIVE_STAGE_DIRS['train'] / 'evaluation_metrics.json')
checks = {
    'D1_real_pretrained_frozen': probe.get('status') == 'PASS' and probe.get('frozen_unchanged') is True,
    'D2_synthetic_and_decoder': probe.get('decoder_updated') is True and (LOCAL_QA_DIR/'synthetic_preview_metadata.json').is_file(),
    'D3_overfit_loss_decreased': overfit.get('overfit_loss_decreased') is True,
    'D3_training_frozen_and_updated': training.get('status') == 'PASS' and training.get('frozen_backbone_unchanged') is True and training.get('decoder_updated') is True,
    'D4_native_DEV_metric': evaluation.get('qa_status') == 'PASS' and evaluation.get('native_resolution') is True and evaluation.get('synthetic_dev_aupro_0_05') is not None,
}
required = ('best.pt','resolved_config.yaml','metrics.json','train.log','loss_curve.png')
checks['D4_required_artifacts'] = all((DRIVE_STAGE_DIRS['train']/name).is_file() for name in required) and bool(list((DRIVE_STAGE_DIRS['train']/'examples').glob('*/predicted_map.npy')))
acceptance = dict(status='PASS' if all(checks.values()) else 'NOT VERIFIED', checks=checks,
                  category=CATEGORY, backbone=BACKBONE_NAME, tile_size=TILE_SIZE, overlap=OVERLAP,
                  context_used=False, downsample_to_512=False, interpretation='synthetic DEV only; not real anomaly performance',
                  drive_run_root=str(DRIVE_RUN_ROOT), gpu_probe=probe)
(LOCAL_QA_DIR/'g1_acceptance.json').write_text(json.dumps(acceptance,indent=2))
copy_atomic(LOCAL_QA_DIR/'g1_acceptance.json', DRIVE_QA_DIR/'g1_acceptance.json')
display(acceptance)
print('Drive outputs:', DRIVE_RUN_ROOT)
print('Synthetic DEV float maps:', DRIVE_STAGE_DIRS['train']/'evaluation/examples')


Output train nằm trong folder đã chọn ở **cell 3**: `best.pt`, `last.pt`, SHA256 sidecars, `resolved_config.yaml`, `launch_config.yaml`, `metrics.json`, `train.log`, `loss_curve.png`, `examples/`. Cell evaluate lưu thêm `evaluation_metrics.json` và `evaluation/examples/` với original, synthetic, exact mask, predicted map `.npy`, PNG và overlay ở đúng native H×W.

`qa/` lưu audit, nguồn code đã nhúng, manifest archive, môi trường, phép đo VRAM thực, synthetic preview và acceptance. Chỉ báo cáo synthetic DEV, không xem đây là hiệu suất trên real anomalies.

Notebook đã được kiểm tra schema, cú pháp, snapshot, workflow archive và CLI bằng kiểm thử local. **Drive mount, L4/A100 và training Colab chưa được thực thi trong môi trường tạo notebook.**
